# Data Collection
### Wikipedia data:
Collects three data types from the MediaWiki API:

    1. Newcomer accounts from the new-user registration log (Special:Log/newusers)

    2. Editing histories per newcomer (to confirm ≥1 edit and compute session edits)

    3. Talk page revision histories per newcomer (to extract messages received)
 
### External dataset
The external dataset (Zheng et al., 2019 bot role classification file) is
loaded from disk — it must be downloaded separately from the project README.
 
Design principles (from course notebook):
    - One shared requests.Session with User-Agent set once.
    - All API calls go through api_get() with maxlag and exponential back-off.
    - Every API response is saved to disk immediately (JSONL append pattern).
    - Re-running the script skips already-collected items without re-hitting the API.
    - A fixed random seed (42) ensures reproducible sampling.
    - The registration log is fetched sequentially; the script stops at
      MAX_LOG_ACCOUNTS (3× the target) and saves progress after every account.



In [5]:
import json
import time
import random
from pathlib import Path
from datetime import datetime, timedelta
import requests
import pandas as pd

In [6]:
API = "https://en.wikipedia.org/w/api.php"

# ── Sampling parameters (from dossier) ───────────────────────────────────────
TARGET_SAMPLE_SIZE = 10_000     # newcomers to analyse
MAX_LOG_ACCOUNTS   = 30_000     # stop fetching the log after this many accounts
RANDOM_SEED        = 42         # fixed for reproducibility

# ── Collection window: March–May 2025 (inclusive), following Zheng et al. (2019)
LOG_START = "2025-03-01T00:00:00Z"
LOG_END   = "2025-05-31T23:59:59Z"

# ── Retention and session windows ────────────────────────────────────────────
RETENTION_DAYS     = 60         # days after first edit to check retention
SESSION_WINDOW_MIN = 60         # minutes defining the first edit session

# ── Polite API call settings (Wikimedia bot policy) ──────────────────────────
# MAXLAG: cooperative DB-lag threshold in seconds. The API returns HTTP 503
# when the replica DB is behind by more than this value, triggering back-off.
# 5 s is the recommended value for non-urgent automated scripts.
MAXLAG              = 5     # seconds; cooperative DB-lag threshold
REQUEST_TIMEOUT     = 15    # seconds before a request is considered hung
RETRY_DELAY         = 2     # initial back-off delay in seconds (doubles each retry)
MAX_RETRY_DELAY     = 60    # back-off is capped at this value to avoid indefinite stalls
MAX_RETRIES         = 5     # maximum retry attempts per API call
INTER_REQUEST_DELAY = 0.1   # polite pause between successive API calls (seconds)

# ── Data directories ─────────────────────────────────────────────────────────
DATA_DIR      = Path.home() / "bdacaII-2026-ada" / "Final_project" / "data"
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
CACHE_DIR     = RAW_DIR / "cache"

for directory in (RAW_DIR, PROCESSED_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# ── JSONL output paths ────────────────────────────────────────────────────────
LOG_JSONL      = RAW_DIR / "registration_log.jsonl"
EDITS_JSONL    = RAW_DIR / "newcomer_edits.jsonl"
TALKPAGE_JSONL = RAW_DIR / "talkpage_messages.jsonl"

# ── Module-level session ──────────────────────────────────────────────────────
# A Session reuses the underlying TCP connection across calls (connection
# pooling) and lets us set the User-Agent header once for all requests.
# Wikimedia policy requires a descriptive User-Agent with contact info.
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "BDACA-II-course/1.0 (educational; adashiyj@gmail.com)"
})

print(f"requests {requests.__version__}")
print(f"pandas   {pd.__version__}")
print(f"Session ready — User-Agent: {SESSION.headers['User-Agent']}")

requests 2.32.5
pandas   2.3.3
Session ready — User-Agent: BDACA-II-course/1.0 (educational; adashiyj@gmail.com)


### Call the API

In [7]:
def api_get(
    params: dict,
    api: str = API,
    max_retries: int = MAX_RETRIES,
    maxlag: int = MAXLAG,
) -> dict:
    """GET the MediaWiki JSON API via the module-level session.

    All API calls in this project go through this wrapper. It handles
    cooperative rate limiting, transient server errors, and API-level
    error responses so that callers never need to implement retry logic
    themselves.

    Parameters
    ----------
    params      : MediaWiki API query parameters (without maxlag).
                  A shallow copy is made internally — the caller's dict
                  is never mutated.
    api         : Base URL of the MediaWiki API endpoint. Defaults to
                  the English Wikipedia API defined at module level.
    max_retries : Maximum number of retry attempts before raising a
                  RuntimeError. Defaults to MAX_RETRIES.
    maxlag      : Cooperative DB-lag threshold in seconds. The API returns
                  HTTP 503 when the replica DB lags beyond this value,
                  which triggers automatic back-off. Defaults to MAXLAG.

    Returns
    -------
    dict
        Parsed JSON response from the API.

    Raises
    ------
    RuntimeError
        If the API returns an error body (even with HTTP 200), or if all
        retry attempts are exhausted.
    """
    # Copy params so we never mutate the caller's dict.
    call_params = {**params, "maxlag": str(maxlag)}

    for attempt in range(1, max_retries + 1):
        try:
            response = SESSION.get(
                api,
                params=call_params,
                timeout=REQUEST_TIMEOUT,
            )

            # Retry on transient server-side errors before raising.
            if response.status_code in (429, 503, 504):
                print(
                    f"  HTTP {response.status_code} "
                    f"(attempt {attempt}/{max_retries}), "
                    f"waiting {RETRY_DELAY}s ..."
                )
                time.sleep(RETRY_DELAY)
                continue

            # Raise immediately for non-retryable HTTP errors (e.g. 404, 400).
            response.raise_for_status()
            result = response.json()

            # The MediaWiki API can return error details inside an HTTP 200
            # response — raise explicitly so callers see a meaningful message.
            if "error" in result:
                err = result["error"]
                raise RuntimeError(
                    f"MediaWiki API error [{err.get('code', '?')}]: "
                    f"{err.get('info', err)}"
                )
            if "errors" in result:
                messages = "; ".join(
                    e.get("text", str(e)) for e in result["errors"]
                )
                raise RuntimeError(f"MediaWiki API errors: {messages}")

            return result

        except requests.exceptions.Timeout:
            # Timeout is retryable — the server may have been temporarily busy.
            print(
                f"  Timeout on attempt {attempt}/{max_retries}, "
                f"waiting {RETRY_DELAY}s ..."
            )
            time.sleep(RETRY_DELAY)

        except requests.exceptions.ConnectionError as exc:
            # Connection errors (e.g. dropped socket) are also retryable.
            print(
                f"  Connection error (attempt {attempt}/{max_retries}): "
                f"{exc!r}, waiting {RETRY_DELAY}s ..."
            )
            time.sleep(RETRY_DELAY)

    raise RuntimeError(
        f"api_get() failed after {max_retries} retries for params: {params}"
    )